In [2]:
import os
import re
from pathlib import Path


# === Date variations ===
DATE_VARIATIONS = [
    r"\d{1,2}(?:st|nd|rd|th)?\s+[A-Za-z]+,?\s*\d{4}",
    r"\d{1,2}\s*[-./]\s*\d{1,2}(?:\s*[-./]\s*\d{2,4})?",
    r"\d{4}\s*[-./]\s*\d{1,2}\s*[-./]\s*\d{1,2}",
    r"\d{1,2}(?:st|nd|rd|th)?\s*[–\-]?\s*[A-Za-z]+\s*[–\-]?\s*\d{4}",
    r"((?:\d\s*){1,2}\s*\.\s*(?:\d\s*){1,2}\s*\.\s*(?:\d\s*){4})",
    r"\d{4}\s*[-./]\s*\d{1,2}\s*(?:[-./]\s*\d{0,2})?",
    r"\d{1,2}\s*[-./]\s*\d{1,2}\s*[-./]\s*\d{2,4}\.?",
    r"\d{4}\s*[–\-]\s*\d{1,2}\s*[–\-]\s*\d{1,2}",
]

# === Metadata phrases ===
METADATA_PHRASES = [
    r"ARGUED\s*&\s*(?:DECIDED)?(?:\s*ON)?",
    r"ARGUED\s*AND\s*(?:DECIDED)?(?:\s*ON)?",
    r"DECIDEN\s*ON",
    r"DECIDED\s*ON",
    r"DECIDEDON",
    r"Decided\s*on",
    r"Decided\s*:?",
    r"Delivered\s*on",
    r"Order\s*(?:Delivered\s*on|on)",
    r"Judgment\s*(?:delivered\s*on|on)",
    r"DATE\s*OF\s*JUDGMENT",
    r"Date",
]

In [ ]:
# === Split function ===
def split_metadata_body(text):
    for phrase in METADATA_PHRASES:
        m = re.search(rf"({phrase}\s*[:;\-–—]*\s*[:;\-–—]*\s*)", text, re.IGNORECASE)
        if m:
            after_phrase = text[m.end() :].lstrip()
            for date_pat in DATE_VARIATIONS:
                date_m = re.match(rf"\s*{date_pat}", after_phrase)
                if date_m:
                    split_idx = m.end() + date_m.end()
                    return text[:split_idx].strip(), text[split_idx:].strip()
            return text[: m.end()].strip(), text[m.end() :].strip()
    return None, text


def process_directory(input_dir: Path, output_dir: Path):
    """Recursively process all .txt files in input_dir and split metadata/body."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    unmatched_files = []
    unmatched_log_path = output_dir / "unmatched_files.txt"

    # Recursively find all .txt files
    txt_files = list(input_dir.rglob("*.txt"))
    print(f"📂 Found {len(txt_files)} text files to process...")

    for input_path in txt_files:
        relative_path = input_path.relative_to(input_dir)
        output_path = output_dir / relative_path
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with open(input_path, "r", encoding="utf-8") as f:
            text = f.read()

        metadata, body = split_metadata_body(text)

        with open(output_path, "w", encoding="utf-8") as f:
            if metadata:
                f.write("=== METADATA ===\n")
                f.write(metadata + "\n\n")
                f.write("=== BODY ===\n")
                f.write(body + "\n")
            else:
                unmatched_files.append(str(relative_path))
                f.write("=== BODY ===\n")
                f.write(text)

    # Write unmatched files log
    if unmatched_files:
        with open(unmatched_log_path, "w", encoding="utf-8") as log_file:
            for file in unmatched_files:
                log_file.write(file + "\n")
        print(f"⚠️ {len(unmatched_files)} file(s) unmatched — see {unmatched_log_path}")
    else:
        print("🎉 All files matched and split successfully!")

    print("✅ Done.")



input_dir = "/kaggle/input/d/muaadhnazly/translated/TRANSLATED"
output_dir = "/kaggle/working/Meta_Splitted"
process_directory(input_dir, output_dir)

📂 Found 1933 text files to process...
🎉 All files matched and split successfully!
✅ Done.


In [ ]:
import os
import re

class BodyLineJoiner:
    def __init__(self):
        pass

    def join_body_lines(self, text: str) -> str:
        # Split at === BODY ===
        parts = re.split(r"(=== BODY ===)", text)
        new_parts = []

        i = 0
        while i < len(parts):
            part = parts[i]
            if part == "=== BODY ===" and i + 1 < len(parts):
                new_parts.append(part)
                body_content = parts[i + 1]
                # Join all lines into a single line
                body_content = " ".join(
                    line.strip() for line in body_content.splitlines() if line.strip()
                )
                new_parts.append(body_content)
                i += 2
            else:
                new_parts.append(part)
                i += 1

        return "\n".join(new_parts)

    def process_file(self, input_file: str, output_file: str):
        with open(input_file, "r", encoding="utf-8") as f:
            content = f.read()
        processed = self.join_body_lines(content)
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(processed)
        print(f"Processed {input_file}")

    def process_directory(self, input_dir: str, output_dir: str):
        """Recursively process all .txt files in input_dir using self.process_file()."""
        input_dir = os.path.abspath(input_dir)
        output_dir = os.path.abspath(output_dir)
        os.makedirs(output_dir, exist_ok=True)

        for root, _, files in os.walk(input_dir):
            for filename in files:
                if not filename.lower().endswith(".txt"):
                    continue

                input_path = os.path.join(root, filename)

                rel_path = os.path.relpath(input_path, input_dir)
                output_path = os.path.join(output_dir, rel_path)
                os.makedirs(os.path.dirname(output_path), exist_ok=True)

                self.process_file(input_path, output_path)

        print("🎉 All files processed!")


INPUT_DIR = "/kaggle/working/Meta_Splitted"
OUTPUT_DIR = "/kaggle/working/Body Joined"
joiner = BodyLineJoiner()
joiner.process_directory(INPUT_DIR, OUTPUT_DIR)


Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_157_2013.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_28_2003.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_61_2024.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_113_2014.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_163_2019.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_113_2011.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_171_2019.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_83_2008.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_88_2016.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_207_2014.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_120_2017.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_50_2010.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_206_2014.txt
Processed /kaggle/working/Meta_Splitted/Civil Appeal/SC_CA_43_2010.tx

In [8]:
!zip -r "/kaggle/working/Body_Joined.zip" "/kaggle/working/Body Joined"

updating: kaggle/working/Body Joined/ (stored 0%)
  adding: kaggle/working/Body Joined/Civil Appeal/ (stored 0%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_157_2013.txt (deflated 69%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_28_2003.txt (deflated 69%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_61_2024.txt (deflated 74%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_113_2014.txt (deflated 71%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_163_2019.txt (deflated 65%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_113_2011.txt (deflated 66%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_171_2019.txt (deflated 71%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_83_2008.txt (deflated 71%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_88_2016.txt (deflated 70%)
  adding: kaggle/working/Body Joined/Civil Appeal/SC_CA_207_2014.txt (deflated 68%)
  adding: kaggle/working/Body Joined/Civil Appeal/S